# Scientific Article Information Retrieval Challenge

This notebook guides you through the challenge step by step:

1. **Section 1 — Setup & Data Exploration**: install packages, load data, understand its structure and statistics.
2. **Section 2 — Helper Functions**: reusable utilities for data loading, chunk extraction, metrics, and evaluation.
3. **Section 3 — TF-IDF Baseline**: sparse retrieval over title + abstract.
4. **Section 4 — Dense Retrieval Baseline**: retrieval using pre-computed dense embeddings.

---

## Task Description

Given a **query** (a scientific paper), retrieve the most relevant documents from a shared **corpus** of 20,000 papers.  
Relevance is defined by **citation**: the gold documents are papers that the query paper cites.

**Submission format**: a JSON file `{query_id: [doc_id_1, ..., doc_id_100]}` — exactly 100 ranked document IDs per query.

**Evaluation**: Recall@10, Recall@100, Precision@10, MRR@10, NDCG@10, NDCG@100, MAP — computed on 100 public queries.

## Environment Setup

Run the following commands **once** from the `ir_challenge/` root folder to create a virtual environment and install all dependencies:

```bash
# Create the virtual environment with Python 3.10
python3.10 -m venv .venv

# Activate it
source .venv/bin/activate          # macOS / Linux
# .venv\Scripts\activate           # Windows

# Install dependencies
pip install -r requirements.txt
```

### Registering the kernel in Jupyter (if not detected automatically)

If Jupyter does not show the environment as a kernel option, run these two extra commands while the venv is activated:

```bash
pip install ipykernel
python -m ipykernel install --user --name ir_challenge --display-name "IR Challenge"
```

Then restart Jupyter and select **IR Challenge** from the kernel menu (`Kernel → Change Kernel`).

---

### VS Code: kernel disappears after trusting the workspace

This is a known VS Code bug — trusting the workspace triggers a re-scan that can drop `.venv` from the kernel list. Try the fixes below in order:

**Fix 1 — Register the kernel explicitly (most reliable)**

Run the two `ipykernel` commands above. This registers it as a Jupyter kernel (not just a Python environment), so it persists across trust changes. Look for it under **Select Another Kernel → Jupyter Kernel...** instead of *Python Environments*.

**Fix 2 — Set the interpreter path manually**

1. Open Command Palette (`Cmd+Shift+P` / `Ctrl+Shift+P`)
2. Type `Python: Select Interpreter`
3. Choose **Enter interpreter path...**
4. Paste: `.venv/bin/python`

Then select the kernel in the notebook — it should now stick.

**Fix 3 — Add interpreter path to VS Code workspace settings**

Open Command Palette → **Preferences: Open Workspace Settings (JSON)** and add:

```json
{
    "python.defaultInterpreterPath": "${workspaceFolder}/.venv/bin/python"
}
```

---
## Folder Structure

```
ir_challenge/
├── data/
│   ├── queries.parquet          # 100 public queries (the papers you retrieve for)
│   ├── corpus.parquet           # 20,000 candidate documents to search over
│   ├── qrels.json               # ground-truth relevance: {query_id: [doc_id, ...]}
│   ├── sample_submission.json   # example of the required submission format
│   └── embeddings/
│       └── sentence-transformers_all-MiniLM-L6-v2/   # pre-computed embeddings
│           ├── query_embeddings.npy   # shape (100, 384)
│           ├── corpus_embeddings.npy  # shape (20000, 384)
│           ├── query_ids.json         # ordered list of query doc_ids
│           ├── corpus_ids.json        # ordered list of corpus doc_ids
│           └── model_info.txt         # model name, dim, normalisation flag
├── notebooks/
│   └── challenge.ipynb          # this notebook
├── scripts/
│   └── embed.py                 # helper to re-encode with any sentence-transformers model
├── submissions/                 # created automatically when you save results
└── requirements.txt
```

### Data files

| File | Rows | Key columns |
|------|------|-------------|
| `queries.parquet` | 100 | `doc_id`, `title`, `abstract`, `ta`, `full_text`, `chunk_meta`, `domain`, `year`, `n_relevant` |
| `corpus.parquet`  | 20,000 | same columns minus `n_relevant` |
| `qrels.json` | 100 queries | `{query_id: [gold_doc_id, ...]}` — a query may have multiple relevant documents |

`n_relevant` tells you how many gold documents for that query are present in the corpus.

### Pre-computed embeddings

The embeddings were produced by [`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) encoding `title + " " + abstract` for every paper. They are **L2-normalised**, so cosine similarity reduces to a dot product.

### `scripts/embed.py` — re-encoding with a different model

If you want to experiment with a stronger encoder, run this from the `ir_challenge/` folder:

```bash
python scripts/embed.py --model BAAI/bge-small-en-v1.5
```

The script automatically creates a model-named subdirectory under `data/embeddings/` so different models never overwrite each other:

```
data/embeddings/
├── sentence-transformers_all-MiniLM-L6-v2/   ← pre-computed (shipped)
└── BAAI_bge-small-en-v1.5/                   ← your new model
```

To use a custom output path explicitly:

```bash
python scripts/embed.py --model BAAI/bge-small-en-v1.5 --output data/embeddings/my_model/
```

Full options: `python scripts/embed.py --help`

---
## Section 1 — Setup & Data Exploration

In [4]:
!pip install pyarrow

ERROR: Could not find a version that satisfies the requirement pyarrow (from versions: none)
ERROR: No matching distribution found for pyarrow


In [3]:
import json
import math
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

# Paths — adjust if your layout differs
DATA_DIR = Path("../data")

# Verify files exist
for f in ["queries.parquet", "corpus.parquet", "qrels.json", "sample_submission.json"]:
    path = DATA_DIR / f
    status = "✓" if path.exists() else "✗ MISSING"
    print(f"  {status}  {path}")

ModuleNotFoundError: No module named 'pyarrow.vendored'

In [ ]:
# Load data
queries = pd.read_parquet(DATA_DIR / "queries.parquet")
corpus  = pd.read_parquet(DATA_DIR / "corpus.parquet")
with open(DATA_DIR / "qrels.json") as f:
    qrels = json.load(f)

print(f"Queries : {len(queries):,} rows — columns: {queries.columns.tolist()}")
print(f"Corpus  : {len(corpus):,} rows  — columns: {corpus.columns.tolist()}")
print(f"Qrels   : {len(qrels):,} queries with gold annotations")

In [ ]:
# Sample query — all available fields
q = queries.iloc[0]
print(f"doc_id     : {q['doc_id']}")
print(f"domain     : {q['domain']}")
print(f"year       : {q['year']}")
print(f"n_relevant : {q['n_relevant']}")
print(f"\nTitle      : {q['title'][:120]}")
print(f"\nAbstract   : {q['abstract'][:300]} ...")
print(f"\nFull-text length (chars) : {len(q['full_text']):,}")
print(f"Chunk meta entries       : {len(json.loads(q['chunk_meta']))}")

In [ ]:
# Show a query and its gold documents side by side
qid = queries.iloc[0]["doc_id"]
gold_ids = qrels.get(qid, [])
gold_docs = corpus[corpus["doc_id"].isin(gold_ids)]

print("=" * 80)
print("QUERY")
print("=" * 80)
print(queries.iloc[0]["title"])
print(f"\n{queries.iloc[0]['abstract'][:300]}...")

print(f"\n=== GOLD DOCUMENTS ({len(gold_ids)} total) ===")
for _, row in gold_docs.head(3).iterrows():
    print(f"\n  [{row['doc_id']}]")
    print(f"  {row['title']}")
    print(f"  {row['abstract'][:200]}...")

In [ ]:
# ── TODO 1 ───────────────────────────────────────────────────────────────────
# Pick a query from a domain of your choice and inspect it.
# Print its title and the titles of its gold documents.
# Do the gold documents share keywords or themes with the query?
#
# Hint: queries["domain"].unique() lists available domains.
#       Use qrels[qid] to get the gold doc IDs for a given query ID.
# ─────────────────────────────────────────────────────────────────────────────

# YOUR CODE HERE


In [ ]:
# ── About this dataset ────────────────────────────────────────────────────
#
# This challenge dataset is a small, curated subset of a larger citation-based
# evaluation benchmark we built from Semantic Scholar (covering 63K papers across
# 19 scientific domains). Queries are citing papers; gold documents are the papers
# they cite. The full evaluation set has ~10K queries — here you work with 100.
#
# Data creation pipeline (briefly):
#   1. Papers were filtered by venue, abstract presence, and minimum text length.
#   2. Citation pairs (A cites B) were extracted and bidirectionally deduplicated.
#   3. A held-out evaluation set was stratified-sampled across all 19 domains,
#      ensuring zero paper overlap with the training split.
#   4. 100 queries were sampled from this eval set (proportional domain coverage),
#      and a 20,000-document corpus was built by combining their gold citations
#      with randomly sampled negatives from the same paper pool.
#
# Each paper exposes: title, abstract (TA), full_text, and chunk_meta.
# chunk_meta encodes pre-computed section boundaries for chunk-level retrieval.
# ─────────────────────────────────────────────────────────────────────────
print("See comments above for the data creation pipeline summary.")

### 1.1 — Domain & Year Distributions

In [ ]:
# Domain distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df, title in [
    (axes[0], queries, "Queries (100)"),
    (axes[1], corpus,  "Corpus (20,000)"),
]:
    counts = df["domain"].value_counts()
    counts.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(title)
    ax.set_xlabel("Count")
    ax.invert_yaxis()
plt.suptitle("Domain Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Year distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, df, title in [
    (axes[0], queries, "Queries"),
    (axes[1], corpus,  "Corpus"),
]:
    years = df["year"].replace(0, np.nan).dropna()
    ax.hist(years, bins=30, color="teal", edgecolor="white")
    ax.set_title(f"Publication Year — {title}")
    ax.set_xlabel("Year")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

### 1.3 — Venue Statistics

In [ ]:
# Venue statistics
n_unique_corpus  = corpus["venue"].replace("", pd.NA).dropna().nunique()
n_unique_queries = queries["venue"].replace("", pd.NA).dropna().nunique()
print(f"Unique venues — corpus: {n_unique_corpus:,}   queries: {n_unique_queries:,}")

# Top 20 venues in corpus
top_venues = (
    corpus["venue"]
    .replace("", pd.NA)
    .dropna()
    .value_counts()
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 6))
top_venues.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 20 Venues — Corpus (20,000 docs)")
ax.set_xlabel("# papers")
ax.invert_yaxis()
for i, v in enumerate(top_venues):
    ax.text(v + 1, i, str(v), va="center", fontsize=8)
plt.tight_layout()
plt.show()

# Top venues per domain (top 3 venues for the 5 largest domains)
print("\nTop 3 venues per domain (5 largest domains):")
top5_domains = corpus["domain"].value_counts().head(5).index
for domain in top5_domains:
    top3 = (
        corpus[corpus["domain"] == domain]["venue"]
        .replace("", pd.NA).dropna()
        .value_counts().head(3)
    )
    print(f"\n  {domain}:")
    for venue, count in top3.items():
        print(f"    {count:>4}  {venue}")

### 1.2 — Citation & Relevance Stats

In [ ]:
# Relevance distribution (how many gold docs per query?)
n_relevant = queries["n_relevant"]
print("Gold docs per query:")
print(f"  mean={n_relevant.mean():.2f}  median={n_relevant.median():.0f}  "
      f"min={n_relevant.min()}  max={n_relevant.max()}")
print(f"  Queries with 1 gold  : {(n_relevant == 1).sum()}")
print(f"  Queries with 2–4     : {((n_relevant >= 2) & (n_relevant <= 4)).sum()}")
print(f"  Queries with 5+      : {(n_relevant >= 5).sum()}")

fig, ax = plt.subplots(figsize=(8, 3))
n_relevant.hist(bins=range(1, n_relevant.max() + 2), ax=ax, color="coral", edgecolor="white")
ax.set_title("Number of Gold Documents per Query")
ax.set_xlabel("# relevant docs in corpus")
ax.set_ylabel("# queries")
plt.tight_layout()
plt.show()

In [ ]:
# Verify all gold docs are present in corpus
corpus_id_set = set(corpus["doc_id"])
coverage = [
    sum(1 for d in docs if d in corpus_id_set) / len(docs)
    for docs in qrels.values() if docs
]
print(f"Average gold coverage in corpus : {np.mean(coverage):.1%}")
print(f"Queries with full coverage      : {sum(1 for c in coverage if c == 1.0)} / {len(coverage)}")

### 1.3 — Title + Abstract Length Distribution

In [ ]:
# TA length stats
for df, label in [(queries, "Queries"), (corpus, "Corpus")]:
    ta_len = (df["title"].fillna("") + " " + df["abstract"].fillna("")).str.len()
    print(f"{label} TA length (chars):  mean={ta_len.mean():.0f}  "
          f"median={ta_len.median():.0f}  p5={ta_len.quantile(0.05):.0f}  "
          f"p95={ta_len.quantile(0.95):.0f}  max={ta_len.max():.0f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, df, label in [(axes[0], queries, "Queries"), (axes[1], corpus, "Corpus")]:
    ta_len = (df["title"].fillna("") + " " + df["abstract"].fillna("")).str.len()
    ax.hist(ta_len, bins=40, color="mediumpurple", edgecolor="white")
    ax.set_title(f"TA Length — {label}")
    ax.set_xlabel("Characters")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

### 1.4 — Full-Text & Chunk Statistics

Each paper includes a `full_text` field (body text with paragraphs separated by `\n\n`) and a `chunk_meta` field encoding pre-computed section boundaries.  
Each entry has a `type` (`"ta"` or `"body"`) and a `char_start` offset into `full_text`.  
The `ta` entry additionally has `char_end`, so it can be sliced directly: `full_text[char_start:char_end]`.  
Body entries only have `char_start` — a section runs from its `char_start` to the next entry's `char_start` (or the end of `full_text` for the last section). The `get_chunks()` helper in Section 2 handles this automatically.

In [ ]:
# Parse chunk_meta for all corpus docs
def parse_chunk_meta(cm_str):
    return json.loads(cm_str) if isinstance(cm_str, str) and cm_str else []

corpus_chunks = corpus["chunk_meta"].apply(parse_chunk_meta)
query_chunks  = queries["chunk_meta"].apply(parse_chunk_meta)

# Number of body chunks per paper (exclude the TA entry)
def n_body_chunks(meta):
    return sum(1 for e in meta if e.get("type") == "body")

corpus_n_body = corpus_chunks.apply(n_body_chunks)
query_n_body  = query_chunks.apply(n_body_chunks)

for label, s in [("Corpus", corpus_n_body), ("Queries", query_n_body)]:
    print(f"{label} body chunks/paper:  mean={s.mean():.1f}  "
          f"median={s.median():.0f}  min={s.min()}  max={s.max()}  "
          f"p5={s.quantile(0.05):.0f}  p95={s.quantile(0.95):.0f}")

In [ ]:
# Distribution of body chunk counts
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, s, label in [
    (axes[0], corpus_n_body, "Corpus"),
    (axes[1], query_n_body,  "Queries"),
]:
    ax.hist(s, bins=range(0, s.max() + 2), color="darkorange", edgecolor="white")
    ax.set_title(f"Body Chunks per Paper — {label}")
    ax.set_xlabel("# body chunks")
    ax.set_ylabel("# papers")
plt.tight_layout()
plt.show()

In [ ]:
# Inspect chunks for a single paper
sample_row = corpus.iloc[0]
meta = parse_chunk_meta(sample_row["chunk_meta"])
full_text = sample_row["full_text"]

print(f"Paper: {sample_row['doc_id']}")
print(f"Title: {sample_row['title'][:80]}")
print(f"Total chunk entries: {len(meta)}  (1 TA + {len(meta)-1} body)\n")

for i, entry in enumerate(meta[:4]):   # show first 4 entries
    char_start = entry["char_start"]
    char_end   = entry.get("char_end") or (
        meta[i + 1]["char_start"] if i + 1 < len(meta) else len(full_text)
    )
    text = full_text[char_start:char_end].strip()
    print(f"[{i}] {entry['type'].upper()} chunk  chars {char_start}–{char_end}  ({char_end - char_start} chars)")
    print(f"    {text[:300]} ...")
    print()

In [ ]:
# Full-text length distribution (corpus)
ft_len = corpus["full_text"].str.len()
print(f"Corpus full_text length (chars): mean={ft_len.mean():.0f}  "
      f"median={ft_len.median():.0f}  p5={ft_len.quantile(0.05):.0f}  "
      f"p95={ft_len.quantile(0.95):.0f}")

fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(ft_len.clip(upper=100_000), bins=50, color="slateblue", edgecolor="white")
ax.set_title("Full-Text Length — Corpus (clipped at 100K chars)")
ax.set_xlabel("Characters")
ax.set_ylabel("# papers")
plt.tight_layout()
plt.show()

In [ ]:
# Body chunks per paper broken down by domain
corpus["n_body_chunks"] = corpus["chunk_meta"].apply(
    lambda cm: sum(1 for e in json.loads(cm) if e.get("type") == "body")
)

domain_order = corpus.groupby("domain")["n_body_chunks"].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(13, 5))
data_by_domain = [corpus[corpus["domain"] == d]["n_body_chunks"].values for d in domain_order]
ax.boxplot(data_by_domain, labels=domain_order, patch_artist=True,
           boxprops=dict(facecolor="lightskyblue"), medianprops=dict(color="navy", linewidth=2))
ax.set_title("Body Chunks per Paper by Domain — Corpus")
ax.set_ylabel("# body chunks")
ax.set_xticklabels(domain_order, rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Median body chunks and section length per domain
print(f"\n{'Domain':<28}  {'Papers':>6}  {'Median chunks':>13}  {'Median section (chars)':>22}")
print("-" * 75)
for domain in domain_order:
    sub = corpus[corpus["domain"] == domain]
    med_chunks = sub["n_body_chunks"].median()
    section_lens = []
    for row in sub.itertuples():
        meta = json.loads(row.chunk_meta)
        body = [e for e in meta if e.get("type") == "body"]
        for j, e in enumerate(body):
            end = body[j + 1]["char_start"] if j + 1 < len(body) else len(row.full_text)
            section_lens.append(end - e["char_start"])
    med_sec = int(np.median(section_lens)) if section_lens else 0
    print(f"{domain:<28}  {len(sub):>6}  {med_chunks:>13.0f}  {med_sec:>22}")

---
## Section 2 — Helper Functions

All reusable utilities are defined here. **Run this cell before any retrieval section.**

In [ ]:
# ════════════════════════════════════════════════════════════
#  HELPER FUNCTIONS — run this block before any retrieval
# ════════════════════════════════════════════════════════════

import json, math, os
from pathlib import Path
import numpy as np
import pandas as pd

# ── Data loaders ────────────────────────────────────────────

def load_queries(path) -> pd.DataFrame:
    """Load queries.parquet → DataFrame with doc_id, title, abstract, ta, full_text, chunk_meta, domain, year, n_relevant."""
    return pd.read_parquet(path)


def load_corpus(path) -> pd.DataFrame:
    """Load corpus.parquet → DataFrame with doc_id, title, abstract, ta, full_text, chunk_meta, domain, year."""
    return pd.read_parquet(path)


def load_qrels(path) -> dict:
    """Load qrels.json → {query_id: [doc_id, ...]}"""
    with open(path) as f:
        return json.load(f)


def load_embeddings(emb_path, ids_path):
    """
    Load pre-computed embeddings and their corresponding IDs.
    Returns (embeddings: np.ndarray float32, ids: list).
    """
    embeddings = np.load(emb_path).astype(np.float32)
    with open(ids_path) as f:
        ids = json.load(f)
    assert len(embeddings) == len(ids), "Embedding count mismatch"
    return embeddings, ids


# ── Text formatting ──────────────────────────────────────────

def format_text(row) -> str:
    """Return title + abstract as a single string for retrieval."""
    title    = str(row.get("title", "") or "").strip()
    abstract = str(row.get("abstract", "") or "").strip()
    if title and abstract:
        return title + " " + abstract
    return title or abstract


# ── Chunk extraction ─────────────────────────────────────────

def get_chunks(full_text: str, chunk_meta_json) -> list:
    """
    Extract all text sections from a paper using pre-computed chunk metadata.

    Each chunk entry has char_start and (for body chunks) the end is inferred
    as the start of the next section, or the end of full_text for the last one.

    Args:
        full_text:       raw paper text (paragraphs separated by \\n\\n)
        chunk_meta_json: JSON string or list of chunk entries from chunk_meta

    Returns:
        list of dicts: [{"type": "ta"|"body", "text": str,
                         "char_start": int, "char_end": int}]
    """
    meta = json.loads(chunk_meta_json) if isinstance(chunk_meta_json, str) else chunk_meta_json
    chunks = []
    for i, entry in enumerate(meta):
        char_start = entry["char_start"]
        if entry["type"] == "ta":
            char_end = entry["char_end"]
        else:
            # Body section spans from char_start to the next section's start (or end of text)
            char_end = meta[i + 1]["char_start"] if i + 1 < len(meta) else len(full_text)
        text = full_text[char_start:char_end].strip()
        chunks.append({"type": entry["type"], "text": text,
                       "char_start": char_start, "char_end": char_end})
    return chunks


def get_ta(row) -> str:
    """Return the pre-extracted title+abstract string from a paper row."""
    return str(row.get("ta", "") or "").strip()


def get_body_chunks(row, min_chars: int = 100) -> list:
    """
    Return all body section texts for a paper row, filtering out very short sections.

    Args:
        row:       a DataFrame row with full_text and chunk_meta columns
        min_chars: minimum character length to keep a section (default 100)
    """
    chunks = get_chunks(row["full_text"], row["chunk_meta"])
    return [c["text"] for c in chunks
            if c["type"] == "body" and len(c["text"]) >= min_chars]


# ── Metric functions (per-query) ─────────────────────────────

def recall_at_k(ranked: list, relevant: set, k: int) -> float:
    """Fraction of relevant docs found in the top-k results."""
    if not relevant:
        return 0.0
    hits = sum(1 for doc in ranked[:k] if doc in relevant)
    return hits / len(relevant)


def precision_at_k(ranked: list, relevant: set, k: int) -> float:
    """Fraction of top-k results that are relevant."""
    if k == 0:
        return 0.0
    hits = sum(1 for doc in ranked[:k] if doc in relevant)
    return hits / k


def mrr_at_k(ranked: list, relevant: set, k: int) -> float:
    """Reciprocal rank of the first relevant document in top-k."""
    for rank, doc in enumerate(ranked[:k], start=1):
        if doc in relevant:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(ranked: list, relevant: set, k: int) -> float:
    """Normalized Discounted Cumulative Gain at k."""
    dcg = sum(
        1.0 / math.log2(rank + 1)
        for rank, doc in enumerate(ranked[:k], start=1)
        if doc in relevant
    )
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / math.log2(r + 1) for r in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def average_precision(ranked: list, relevant: set) -> float:
    """Average Precision (area under the precision-recall curve) for one query."""
    if not relevant:
        return 0.0
    hits, score = 0, 0.0
    for rank, doc in enumerate(ranked, start=1):
        if doc in relevant:
            hits += 1
            score += hits / rank
    return score / len(relevant)


# ── Aggregate evaluation ─────────────────────────────────────

def evaluate(
    submission: dict,
    qrels: dict,
    ks: list = None,
    query_domains: dict = None,
    verbose: bool = True,
) -> dict:
    """
    Evaluate a retrieval submission.

    Args:
        submission:    {query_id: [top-100 doc_ids]}
        qrels:         {query_id: [relevant doc_ids]}
        ks:            cut-off values, default [10, 100]
        query_domains: {query_id: domain} — enables per-domain breakdown
        verbose:       print formatted results table

    Returns:
        dict with 'overall' metrics and optional 'per_domain'
    """
    if ks is None:
        ks = [10, 100]

    per_query = {}
    for qid, rel_list in qrels.items():
        relevant = set(rel_list)
        ranked   = submission.get(qid, [])
        q = {}
        for k in ks:
            q[f"Recall@{k}"]    = recall_at_k(ranked, relevant, k)
            q[f"Precision@{k}"] = precision_at_k(ranked, relevant, k)
            q[f"MRR@{k}"]       = mrr_at_k(ranked, relevant, k)
            q[f"NDCG@{k}"]      = ndcg_at_k(ranked, relevant, k)
        q["AP"] = average_precision(ranked, relevant)
        per_query[qid] = q

    metric_keys = list(next(iter(per_query.values())).keys()) if per_query else []
    overall = {}
    for key in metric_keys:
        vals = [per_query[qid][key] for qid in per_query]
        overall[key] = float(np.mean(vals))
    overall["MAP"] = overall.pop("AP", 0.0)
    overall["num_queries"] = len(per_query)

    result = {"overall": overall, "per_query": per_query}

    if query_domains:
        per_domain = {}
        for domain in sorted(set(query_domains.values())):
            dqids = [q for q in per_query if query_domains.get(q) == domain]
            if not dqids:
                continue
            dm = {}
            for key in metric_keys:
                dm[key] = float(np.mean([per_query[q][key] for q in dqids]))
            dm["MAP"] = dm.pop("AP", 0.0)
            dm["num_queries"] = len(dqids)
            per_domain[domain] = dm
        result["per_domain"] = per_domain

    if verbose:
        _print_results(result, ks)

    return result


def _print_results(results: dict, ks: list):
    o = results["overall"]
    print("\n" + "=" * 68)
    print("OVERALL RESULTS")
    print("=" * 68)
    for label, keys in [
        ("Recall",    [f"Recall@{k}"    for k in ks]),
        ("Precision", [f"Precision@{k}" for k in ks]),
        ("MRR",       [f"MRR@{k}"       for k in ks]),
        ("NDCG",      [f"NDCG@{k}"      for k in ks]),
    ]:
        row = f"{label:<14}"
        for k, key in zip(ks, keys):
            row += f"  @{k:>3}: {o.get(key, 0):.4f}"
        print(row)
    print(f"{'MAP':<14}  {o.get('MAP', 0):.4f}")
    print(f"{'Queries':<14}  {int(o.get('num_queries', 0))}")

    if "per_domain" in results:
        print("\n" + "-" * 68)
        print("PER-DOMAIN  (first k only)")
        print("-" * 68)
        k = ks[0]
        print(f"  {'Domain':<28} R@{k:<3} P@{k:<3} MRR@{k:<3} NDCG@{k:<3}  MAP    n")
        for domain, dm in sorted(results["per_domain"].items()):
            print(
                f"  {domain:<28}"
                f" {dm.get(f'Recall@{k}', 0):.3f}"
                f" {dm.get(f'Precision@{k}', 0):.3f}"
                f" {dm.get(f'MRR@{k}', 0):.3f}  "
                f" {dm.get(f'NDCG@{k}', 0):.3f}"
                f"  {dm.get('MAP', 0):.3f}"
                f"  {int(dm.get('num_queries', 0))}"
            )
    print()


print("Helper functions loaded.")

---
## Section 3 — TF-IDF Baseline

A simple sparse retrieval baseline using TF-IDF cosine similarity over **title + abstract** only.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load data (re-load in case this section is run independently)
DATA_DIR = Path("../data")
queries = load_queries(DATA_DIR / "queries.parquet")
corpus  = load_corpus(DATA_DIR / "corpus.parquet")
qrels   = load_qrels(DATA_DIR / "qrels.json")

query_ids  = queries["doc_id"].tolist()
corpus_ids = corpus["doc_id"].tolist()

# Use title + abstract for retrieval (TA only)
query_texts  = [format_text(row) for _, row in queries.iterrows()]
corpus_texts = [format_text(row) for _, row in corpus.iterrows()]

print(f"Queries : {len(query_texts)}")
print(f"Corpus  : {len(corpus_texts)}")

In [ ]:
# Build TF-IDF index on corpus TA
vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 1),
    stop_words="english",
)
corpus_matrix = vectorizer.fit_transform(corpus_texts)   # (20000, vocab)
query_matrix  = vectorizer.transform(query_texts)        # (100, vocab)
print(f"Corpus matrix : {corpus_matrix.shape}  vocab={corpus_matrix.shape[1]:,}")
print(f"Query  matrix : {query_matrix.shape}")

In [ ]:
# Retrieve top-100 per query
TOP_K = 100
sim_matrix = cosine_similarity(query_matrix, corpus_matrix)   # (100, 20000)

tfidf_submission = {}
for i, qid in enumerate(query_ids):
    top_indices = np.argsort(-sim_matrix[i])[:TOP_K]
    tfidf_submission[qid] = [corpus_ids[j] for j in top_indices]

print(f"Submission: {len(tfidf_submission)} queries × {TOP_K} results")

In [ ]:
# Evaluate
query_domains = dict(zip(queries["doc_id"], queries["domain"]))
tfidf_results = evaluate(tfidf_submission, qrels, ks=[10, 100],
                         query_domains=query_domains, verbose=True)

In [ ]:
# ── TODO 2 ───────────────────────────────────────────────────────────────────
# Pick any query and inspect its top-5 TF-IDF results.
# For each retrieved document, print its title and whether it is a gold doc.
#
# Hint: tfidf_submission[qid] gives the ranked list of doc IDs for that query.
#       qrels[qid] gives the set of gold doc IDs.
# ─────────────────────────────────────────────────────────────────────────────

# YOUR CODE HERE


In [ ]:
# Save submission
os.makedirs("../submissions", exist_ok=True)
with open("../submissions/tfidf_baseline.json", "w") as f:
    json.dump(tfidf_submission, f)
print("Saved → ../submissions/tfidf_baseline.json")

---
## Section 4 — Dense Retrieval Baseline

Dense retrieval using pre-computed embeddings from [`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) (22M params).  
Texts were encoded as `title + " " + abstract`. Embeddings are L2-normalised → cosine similarity = dot product.  
To re-encode with a different model, run `scripts/embed.py --model <model-name>`.

In [ ]:
EMB_DIR = DATA_DIR / "embeddings" / "sentence-transformers_all-MiniLM-L6-v2"

for f in ["query_embeddings.npy", "corpus_embeddings.npy",
          "query_ids.json", "corpus_ids.json", "model_info.txt"]:
    path = EMB_DIR / f
    status = "✓" if path.exists() else "✗ MISSING"
    print(f"  {status}  {path}")

if (EMB_DIR / "model_info.txt").exists():
    print("\nModel info:")
    print((EMB_DIR / "model_info.txt").read_text())

In [ ]:
# Load embeddings
query_embs, q_ids = load_embeddings(EMB_DIR / "query_embeddings.npy",
                                     EMB_DIR / "query_ids.json")
corpus_embs, c_ids = load_embeddings(EMB_DIR / "corpus_embeddings.npy",
                                      EMB_DIR / "corpus_ids.json")
print(f"Query  embeddings : {query_embs.shape}")
print(f"Corpus embeddings : {corpus_embs.shape}")

In [ ]:
# Dense retrieval
TOP_K = 100

# ── TODO 3 ───────────────────────────────────────────────────────────────────
# Compute the cosine similarity matrix between all queries and all corpus docs.
# The embeddings are already L2-normalised, so cosine similarity = dot product.
# sim_matrix should have shape (n_queries, n_corpus).
#
# Hint: use matrix multiplication between query_embs and corpus_embs.
# ─────────────────────────────────────────────────────────────────────────────

sim_matrix = None  # YOUR CODE HERE

top_indices = np.argsort(-sim_matrix, axis=1)[:, :TOP_K]          # (100, 100)

dense_submission = {qid: [c_ids[j] for j in top_indices[i]]
                    for i, qid in enumerate(q_ids)}
print(f"Submission: {len(dense_submission)} queries × {TOP_K} results")

In [ ]:
# Evaluate
dense_results = evaluate(dense_submission, qrels, ks=[10, 100],
                         query_domains=query_domains, verbose=True)

In [ ]:
# Summary comparison
comparison = []
for name, res in [("TF-IDF", tfidf_results), ("all-MiniLM-L6-v2", dense_results)]:
    row = {"Model": name}
    row.update({k: f"{v:.4f}" for k, v in res["overall"].items() if k != "num_queries"})
    comparison.append(row)
pd.DataFrame(comparison).set_index("Model")

In [ ]:
# Save dense submission
with open("../submissions/dense_minilm.json", "w") as f:
    json.dump(dense_submission, f)
print("Saved → ../submissions/dense_minilm.json")

---
## What's Next?

You now have two baselines over title + abstract. The data also exposes `full_text` and `chunk_meta` for more advanced retrieval.

### Sparse retrieval improvements
- **BM25**: tune `k1` and `b` parameters; typically outperforms TF-IDF on recall.
- **Full-text BM25**: index body chunks instead of TA — use `get_body_chunks(row)` from the helpers.

### Dense retrieval improvements
- **Domain-specific models**: try `allenai/specter2_base` (trained on scientific citations).
- **Chunk-level encoding**: encode each body chunk separately, aggregate scores per document (max or mean pooling).
- **Re-encode**: `scripts/embed.py --model <model-name>` generates fresh embedding files.

### Hybrid retrieval
- **Reciprocal Rank Fusion (RRF)**: combine ranked lists without score normalisation.
  ```python
  # RRF score for doc d = Σ_i  1 / (60 + rank_i(d))
  ```
- **Score interpolation**: `α * sparse_score + (1-α) * dense_score` after per-retriever normalisation.

### Reranking
- **Cross-encoder reranker**: rerank the top-100 results from a first-stage retriever.
- **Paragraph-level reranking**: score each body chunk independently, aggregate to document level.

Submit your best system as `{query_id: [top-100 doc_ids]}`.